In [1]:
!pip install numpy==1.26.4
!pip install opencv-python==4.10.0.84
!pip install paddlepaddle
!pip install paddleocr

  Using cached opencv_python-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached opencv_python-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (62.5 MB)


In [38]:
from paddleocr import PaddleOCR
# Initializing the Optical Charcter Recognition Engine for the project
ocr = PaddleOCR(
    use_angle_cls=True,
    lang='en'
)

/tmp/ipykernel_14234/3582383596.py:3: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models

In [12]:
import cv2
# Converting image to gray scale preprocessing the image
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    threshold = cv2.adaptiveThreshold(gray,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,11,2)
    return threshold

In [16]:
# Extracting text from the image
def extract_text(image_path):
    processed = preprocess_image(image_path)
    temp_file = "temp_receipt.png"
    cv2.imwrite(temp_file, processed)
    result = ocr.predict(temp_file)
    text_lines = []
    if result and result[0]:
        for line in result[0]:
            text_lines.append(line[1][0])
    return "\n".join(text_lines)

In [9]:
# Extracting the required feild form the image
def extract_details(text):
    data = {
        "date": "Not Found",
        "time": "Not Found",
        "bill_number": "Not Found",
        "items": []
    }
    # Date
    date_match = re.search(
        r"\d{2}[-/]\d{2}[-/]\d{4}",
        text
    )
    if date_match:
        data["date"] = date_match.group()
    # Time
    time_match = re.search(
        r"\d{2}:\d{2}(:\d{2})?",
        text
    )
    if time_match:
        data["time"] = time_match.group()
    # Bill Number
    bill_match = re.search(
        r"(Bill|Invoice)\s*(No|#)?\s*[:\-]?\s*([A-Za-z0-9\-]+)",
        text,
        re.IGNORECASE
    )
    if bill_match:
        data["bill_number"] = bill_match.group(3)
    # Items
    lines = text.split("\n")
    for line in lines:
        if re.search(r"\d+\.\d{2}", line):
            data["items"].append(line)
    return data

In [31]:
# Preprocessing the receipt image
def process_receipt(image_path):
    text = extract_text(image_path)
    details = extract_details(text)
    print("\n===== RECEIPT DETAILS =====")
    print("Date       :", details["date"])
    print("Time       :", details["time"])
    print("Bill No    :", details["bill_number"])
    print("\nItems:")
    for item in details["items"]:
        print("-", item)
    with open("receipt_output.json", "w") as file:
        json.dump(details, file, indent=4)
    print("\nResults saved to receipt_output.json")

In [39]:
# Opening the image that is to be preprocessed and should give the result
image_path="sample_receipt.jpg"
process_receipt(image_path)

NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at /paddle/paddle/fluid/framework/new_executor/instruction/onednn/onednn_instruction.cc:116)
